# Qwen3.5-9B 4-bit LoRA VQA Baseline — RTX 5060 Ti 16GB

기존 `(260827)_baseline_desktop5060ti`의 흐름을 유지하면서 Qwen3.5-9B 기반으로 다시 구성한 버전입니다.

- Qwen3.5-9B: 네이티브 멀티모달 모델
- 4-bit NF4 양자화 + LoRA
- RTX 5060 Ti 16GB 기준 메모리 절약 설정
- `data/train`, `data/test` 경로를 자동 탐색
- 학습 데이터/에폭/LoRA/학습률/이미지 크기를 **상단 CONFIG 셀에서 한 번에 조절**
- 검증셋과 best checkpoint 저장
- 테스트 6,714장 수준의 대량 추론을 고려한 결과 저장
- OCR(PaddleOCR)은 사용하지 않음


## 0. 먼저 확인할 것

이 노트북은 현재 사용 중인 Windows 환경의 실제 데이터 구조를 기준으로 합니다.

예상 구조:

```text
C:\Users\SSAFY\Downloads\
├─ data\
│  ├─ train\
│  └─ test\
├─ train.csv
├─ test.csv
└─ (이 노트북).ipynb
```

CSV의 이미지 경로가 `train/train_0001.jpg`처럼 되어 있어도 자동으로 `data/train/train_0001.jpg`를 찾도록 처리합니다.


In [1]:
# ============================================================
# 1. 설치 / 라이브러리
# ============================================================
# Qwen3.5는 최신 Transformers 계열을 사용합니다.
# 커널을 재시작하라는 메시지가 나오면 재시작 후 이 셀부터 다시 실행하세요.

%pip -q install -U "transformers>=5.13.0" "accelerate>=1.10.0" "peft>=0.18.0" "bitsandbytes>=0.49.0" "pillow>=10.0.0" "pandas>=2.2.0" "tqdm>=4.66.0"


Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# 2. CONFIG — 여기만 바꾸면 주요 실험 설정을 조절할 수 있음
# ============================================================
import os
import re
import math
import random
import gc
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from tqdm.auto import tqdm

# -----------------------------
# 모델
# -----------------------------
MODEL_ID = "Qwen/Qwen3.5-9B"

# -----------------------------
# 데이터
# -----------------------------
TRAIN_CSV = "train.csv"
TEST_CSV  = "test.csv"
DATA_ROOT = "data"

# None = 전체 학습 데이터 사용
# 처음 테스트는 300~500, 실제 학습은 1500~2500 권장
TRAIN_SAMPLES = 2000

# 검증 비율
VALID_RATIO = 0.10

# -----------------------------
# 이미지 / 프롬프트
# -----------------------------
IMAGE_SIZE = 448
MAX_NEW_TOKENS = 2

# -----------------------------
# 학습
# -----------------------------
NUM_EPOCHS = 2

# RTX 5060 Ti 16GB에서 안전한 기본값
BATCH_SIZE = 1
GRAD_ACCUM = 8

LEARNING_RATE = 1.0e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

# -----------------------------
# LoRA
# -----------------------------
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Qwen3.5의 구조가 기존 Qwen2.5-VL과 다르므로
# 특정 모듈명을 하드코딩하지 않고 all-linear를 사용
LORA_TARGET_MODULES = "all-linear"

# -----------------------------
# 양자화
# -----------------------------
USE_4BIT = True
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_4BIT_COMPUTE_DTYPE = torch.bfloat16
BNB_4BIT_USE_DOUBLE_QUANT = True

# -----------------------------
# 기타
# -----------------------------
SEED = 42
NUM_WORKERS = 0  # Windows 안정성 우선
PIN_MEMORY = True

OUTPUT_ROOT = os.path.join("content", "qwen3_5_9b_vqa")
BEST_DIR = os.path.join(OUTPUT_ROOT, "best")
FINAL_DIR = os.path.join(OUTPUT_ROOT, "final")
SUBMISSION_PATH = os.path.join(OUTPUT_ROOT, "submission.csv")

os.makedirs(OUTPUT_ROOT, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

print("\n===== CONFIG =====")
for k in [
    "MODEL_ID", "TRAIN_SAMPLES", "VALID_RATIO", "IMAGE_SIZE",
    "NUM_EPOCHS", "BATCH_SIZE", "GRAD_ACCUM", "LEARNING_RATE",
    "LORA_R", "LORA_ALPHA", "LORA_DROPOUT"
]:
    print(f"{k}: {globals()[k]}")


Device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti
VRAM: 15.9 GB

===== CONFIG =====
MODEL_ID: Qwen/Qwen3.5-9B
TRAIN_SAMPLES: 2000
VALID_RATIO: 0.1
IMAGE_SIZE: 448
NUM_EPOCHS: 2
BATCH_SIZE: 1
GRAD_ACCUM: 8
LEARNING_RATE: 0.0001
LORA_R: 16
LORA_ALPHA: 32
LORA_DROPOUT: 0.05


c:\Users\SSAFY\Downloads\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 3. 데이터 경로 자동 해결
# ============================================================
from pathlib import Path

def first_existing(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

train_csv_path = first_existing([
    TRAIN_CSV,
    os.path.join(DATA_ROOT, "train.csv"),
    os.path.join("content", "train.csv"),
])

test_csv_path = first_existing([
    TEST_CSV,
    os.path.join(DATA_ROOT, "test.csv"),
    os.path.join("content", "test.csv"),
])

if train_csv_path is None:
    raise FileNotFoundError("train.csv를 찾지 못했습니다.")
if test_csv_path is None:
    raise FileNotFoundError("test.csv를 찾지 못했습니다.")

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

print("train csv:", train_csv_path, len(train_df))
print("test csv :", test_csv_path, len(test_df))
print("train columns:", list(train_df.columns))

def resolve_image_path(p):
    p = os.path.normpath(str(p))

    candidates = [
        p,
        os.path.join(DATA_ROOT, p),
        os.path.join(DATA_ROOT, "train", os.path.basename(p)),
        os.path.join(DATA_ROOT, "test", os.path.basename(p)),
        os.path.join("content", p),
    ]

    for c in candidates:
        c = os.path.normpath(c)
        if os.path.isfile(c):
            return c

    raise FileNotFoundError(
        "이미지를 찾을 수 없습니다.\n"
        f"원본: {p}\n"
        f"확인한 후보: {candidates}"
    )

train_df["path"] = train_df["path"].apply(resolve_image_path)
test_df["path"] = test_df["path"].apply(resolve_image_path)

print("\n첫 train:", train_df["path"].iloc[0])
print("존재:", os.path.exists(train_df["path"].iloc[0]))
print("첫 test :", test_df["path"].iloc[0])
print("존재:", os.path.exists(test_df["path"].iloc[0]))


train csv: data\train.csv 6714
test csv : data\test.csv 6714
train columns: ['id', 'path', 'question', 'a', 'b', 'c', 'd', 'answer']

첫 train: data\train\train_0001.jpg
존재: True
첫 test : data\test\test_0001.jpg
존재: True


In [4]:
# ============================================================
# 4. 학습 샘플 선택 + train/valid 분리
# ============================================================
# 정답/필수 컬럼 확인
required_cols = {"path", "question", "a", "b", "c", "d", "answer"}
missing_cols = required_cols - set(train_df.columns)
if missing_cols:
    raise KeyError(f"train.csv에 필요한 컬럼이 없습니다: {missing_cols}")

# 답을 a/b/c/d 형태로 통일
train_df["answer"] = (
    train_df["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.extract(r"([abcd])", expand=False)
)

bad_answer = train_df["answer"].isna().sum()
if bad_answer:
    raise ValueError(f"정답을 a/b/c/d로 변환하지 못한 행이 {bad_answer}개 있습니다.")

train_df = train_df[train_df["answer"].isin(["a", "b", "c", "d"])].reset_index(drop=True)

if TRAIN_SAMPLES is not None:
    n = min(TRAIN_SAMPLES, len(train_df))
    train_df = train_df.sample(n=n, random_state=SEED).reset_index(drop=True)

# 섞은 뒤 train/valid 분리
train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
split = int(len(train_df) * (1.0 - VALID_RATIO))

train_subset = train_df.iloc[:split].reset_index(drop=True)
valid_subset = train_df.iloc[split:].reset_index(drop=True)

print(f"학습: {len(train_subset)}")
print(f"검증: {len(valid_subset)}")
print("\n정답 분포:")
print(train_subset["answer"].value_counts(normalize=True).sort_index())


학습: 1800
검증: 200

정답 분포:
answer
a    0.242778
b    0.241667
c    0.246667
d    0.268889
Name: proportion, dtype: float64


In [5]:
# ============================================================
# 5. Qwen3.5 Processor / Model
# ============================================================
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

print("Loading:", MODEL_ID)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
    bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config if USE_4BIT else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

print("Model loaded.")
print("Model device:", next(model.parameters()).device)


Loading: Qwen/Qwen3.5-9B


c:\Users\SSAFY\Downloads\baseline\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--Qwen--Qwen3.5-9B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\SSAFY\Downloads\baseline\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub

Model loaded.
Model device: cuda:0


In [6]:
# ============================================================
# 6. Prompt
# ============================================================
SYSTEM_INSTRUCT = (
    "You are an expert visual multiple-choice question answering model. "
    "Use the image and the question carefully. "
    "Return exactly one lowercase letter: a, b, c, or d. "
    "Do not explain your answer."
)

def build_mc_prompt(q, a, b, c, d):
    return (
        f"Question: {q}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "Answer with exactly one letter: a, b, c, or d."
    )

print(build_mc_prompt("예시 질문", "선택지1", "선택지2", "선택지3", "선택지4"))


Question: 예시 질문
(a) 선택지1
(b) 선택지2
(c) 선택지3
(d) 선택지4

Answer with exactly one letter: a, b, c, or d.


In [7]:
# ============================================================
# 7. Dataset / Collator
# ============================================================
class VQADataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # Qwen processor가 원본 해상도를 처리하도록 두고,
        # 이미지 토큰 수가 너무 커지는 것을 피하기 위해 최대 크기만 제한
        if max(img.size) > IMAGE_SIZE:
            scale = IMAGE_SIZE / max(img.size)
            new_size = (
                max(1, int(img.width * scale)),
                max(1, int(img.height * scale))
            )
            img = img.resize(new_size, Image.Resampling.LANCZOS)

        q = str(row["question"])
        a, b, c, d = map(str, [row["a"], row["b"], row["c"], row["d"]])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": user_text}
                ]
            }
        ]

        if self.train:
            gold = str(row["answer"])
            messages.append({
                "role": "assistant",
                "content": [{"type": "text", "text": gold}]
            })

        return {"messages": messages, "image": img, "answer": str(row["answer"]) if self.train else None}


def find_subsequence(sequence, pattern):
    if len(pattern) == 0 or len(pattern) > len(sequence):
        return -1
    start = len(sequence) - len(pattern)
    for i in range(start, -1, -1):
        if sequence[i:i+len(pattern)] == pattern:
            return i
    return -1


@dataclass
class VQACollator:
    processor: object
    train: bool = True

    def __call__(self, batch):
        texts = []
        images = []
        answers = []

        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append(text)
            images.append(sample["image"])
            if self.train:
                answers.append(sample["answer"])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        if self.train:
            labels = torch.full_like(enc["input_ids"], -100)

            for i, answer in enumerate(answers):
                answer_ids = self.processor.tokenizer(
                    answer,
                    add_special_tokens=False,
                    return_tensors="pt",
                )["input_ids"][0].tolist()

                ids = enc["input_ids"][i].tolist()
                pos = find_subsequence(ids, answer_ids)

                if pos >= 0:
                    labels[i, pos:pos+len(answer_ids)] = enc["input_ids"][i, pos:pos+len(answer_ids)]
                else:
                    # 마지막 토큰 하나라도 학습할 수 있게 fallback
                    labels[i, -1] = enc["input_ids"][i, -1]

            enc["labels"] = labels

        return enc


train_ds = VQADataset(train_subset, processor, train=True)
valid_ds = VQADataset(valid_subset, processor, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=VQACollator(processor, True),
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=VQACollator(processor, True),
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("train batches:", len(train_loader))
print("valid batches:", len(valid_loader))


train batches: 1800
valid batches: 200


In [8]:
# ============================================================
# 8. LoRA
# ============================================================
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 51,265,024 || all params: 9,461,078,768 || trainable%: 0.5419


In [9]:
# ============================================================
# 9. 학습
# ============================================================
from transformers import get_cosine_schedule_with_warmup

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = NUM_EPOCHS * updates_per_epoch
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

best_val_loss = float("inf")
history = []

print(f"total optimizer steps: {total_steps}")
print(f"warmup steps: {warmup_steps}")

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    update_count = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]",
        unit="batch"
    )

    for step, batch in enumerate(pbar, start=1):
        batch = {
            k: v.to(model.device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            update_count += 1
            avg_loss = running_loss / min(GRAD_ACCUM, step)
            pbar.set_postfix(
                loss=f"{avg_loss:.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}"
            )
            running_loss = 0.0

    # -----------------------------
    # validation
    # -----------------------------
    model.eval()
    val_loss = 0.0
    val_steps = 0

    with torch.no_grad():
        for batch in tqdm(
            valid_loader,
            desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [valid]",
            unit="batch"
        ):
            batch = {
                k: v.to(model.device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=torch.cuda.is_available()
            ):
                out = model(**batch)

            val_loss += out.loss.item()
            val_steps += 1

    val_loss /= max(1, val_steps)
    history.append({
        "epoch": epoch + 1,
        "valid_loss": val_loss,
    })

    print(f"[Epoch {epoch+1}] valid loss = {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs(BEST_DIR, exist_ok=True)
        model.save_pretrained(BEST_DIR)
        processor.save_pretrained(BEST_DIR)
        print("Best checkpoint saved:", BEST_DIR)

print("Training finished.")
print("Best valid loss:", best_val_loss)


total optimizer steps: 450
warmup steps: 22


Epoch 1/2 [train]:   0%|          | 0/1800 [00:00<?, ?batch/s]c:\Users\SSAFY\Downloads\baseline\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\SSAFY\Downloads\baseline\Lib\site-packages\torch\utils\checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
c:\Users\SSAFY\Downloads\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
[transformers] `

[Epoch 1] valid loss = 0.35159
Best checkpoint saved: content\qwen3_5_9b_vqa\best


Epoch 2/2 [valid]: 100%|██████████| 200/200 [39:50<00:00, 11.95s/batch]

[Epoch 2] valid loss = 0.59492
Training finished.
Best valid loss: 0.35158873925444595


In [10]:
# ============================================================
# 10. 최종 LoRA 저장
# ============================================================
os.makedirs(FINAL_DIR, exist_ok=True)
model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)

pd.DataFrame(history).to_csv(
    os.path.join(OUTPUT_ROOT, "history.csv"),
    index=False
)

print("Final:", FINAL_DIR)


Final: content\qwen3_5_9b_vqa\final


In [11]:
# ============================================================
# 11. 추론용 choice 추출
# ============================================================
def extract_choice(text):
    text = str(text).strip().lower()

    # 모델이 긴 문장을 내더라도 a/b/c/d를 우선 추출
    m = re.search(r"(?<![a-z])([abcd])(?![a-z])", text)
    if m:
        return m.group(1)

    # 마지막 방어 로직
    for ch in "abcd":
        if ch in text:
            return ch

    return "a"


def make_inference_messages(row):
    img = Image.open(row["path"]).convert("RGB")

    if max(img.size) > IMAGE_SIZE:
        scale = IMAGE_SIZE / max(img.size)
        new_size = (
            max(1, int(img.width * scale)),
            max(1, int(img.height * scale))
        )
        img = img.resize(new_size, Image.Resampling.LANCZOS)

    prompt = build_mc_prompt(
        row["question"],
        row["a"],
        row["b"],
        row["c"],
        row["d"],
    )

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    return messages, img


model.eval()

# 모델이 이미 LoRA를 포함하고 있으므로 현재 model을 그대로 사용


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1152)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-26): 27 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1152, out_features=3456, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
             

In [12]:
# ============================================================
# 12. 테스트 추론
# ============================================================
preds = []
raw_outputs = []

for i in tqdm(
    range(len(test_df)),
    desc="Inference",
    unit="image"
):
    row = test_df.iloc[i]

    messages, img = make_inference_messages(row)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[img],
        return_tensors="pt",
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=torch.cuda.is_available()
        ):
            out_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

    # 입력 부분 제거하고 새로 생성된 부분만 decode
    input_len = inputs["input_ids"].shape[1]
    generated = out_ids[:, input_len:]

    output_text = processor.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    raw_outputs.append(output_text)
    preds.append(extract_choice(output_text))

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds,
})

submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved:", SUBMISSION_PATH)
print(submission.head())
print("\nPrediction distribution:")
print(submission["answer"].value_counts(normalize=True).sort_index())


Inference:   0%|          | 0/6714 [00:00<?, ?image/s]c:\Users\SSAFY\Downloads\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
Inference:   4%|▍         | 289/6714 [1:08:13<25:16:54, 14.17s/image]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 13. 결과 일부 확인
# ============================================================
result_preview = test_df[["id", "path", "question"]].copy()
result_preview["prediction"] = preds
result_preview["raw_output"] = raw_outputs

display(result_preview.head(20))

print("\nSubmission:", SUBMISSION_PATH)
print("Rows:", len(submission))
